# Análisis de Datos · Semana 15, sesión 3 de 3
## Agrupar, resumir y unir

**TIA502 · Facultad de Empresariales · Profesor David Escobar-Castillejos**

Las ochenta líneas que escribiste a mano en la semana 14 caben hoy en ocho, y los números
coinciden hasta el centavo. Vas a ver la tabla dinámica y el `BUSCARV` escritos, y además
una revisión que ninguna hoja de cálculo te deja hacer.

Al terminar este cuaderno vas a poder:

1. Agrupar y resumir en una línea, con `groupby` seguido de la columna y la función.
2. Pedir varios resúmenes a la vez con `agg`, y con nombres de columna que tú eliges.
3. Armar una rejilla con `pivot_table`, incluidos los totales de fila y de columna.
4. Unir dos tablas con `merge`, y explicar por qué el modo izquierdo es el seguro.
5. Auditar una unión con `indicator`, revisando las dos direcciones antes de confiar.

### Cómo se usa este cuaderno

Ejecuta las celdas en orden. Este cuaderno no depende de que hayas corrido el de la sesión
15.2: la segunda celda repite la limpieza para que puedas abrir este solo.

Tres celdas fallan a propósito y llevan un comentario que lo dice.

---
## Preparación

In [ ]:
import pandas as pd

print("pandas", pd.__version__)

In [ ]:
# Plomería, no lección. Esta celda deja los tres CSV del curso al
# alcance de pandas y no vuelve a hacer falta.
#
# Primero los busca en el repositorio, que es público y se lee por URL.
# Si no responde, los reconstruye aquí mismo con la semilla fija del
# curso, así que salen idénticos por cualquiera de los dos caminos.
# En ningún caso hay que subir un archivo a mano.
import urllib.request
from pathlib import Path

BASE = ("https://raw.githubusercontent.com/Davidowa/learning-hub/main/"
        "docs/en/courses/python-course/06%20-%20Advanced/data/")
ARCHIVOS = ["sales.csv", "regions.csv", "employees.csv"]


def _descargar():
    for nombre in ARCHIVOS:
        with urllib.request.urlopen(BASE + nombre, timeout=15) as r:
            Path(nombre).write_bytes(r.read())


def _reconstruir_datos():
    """Vuelve a escribir los tres CSV con la semilla fija del curso.

    Salen idénticos byte por byte a los del repositorio, así que los
    números de la diapositiva siguen coincidiendo con los del cuaderno.
    """
    import csv, random
    from datetime import date, timedelta

    rng = random.Random(20260808)
    REGIONS = ["North", "South", "Centre", "West"]
    CHANNELS = ["Retail", "Online", "Wholesale"]
    PRODUCTS = {"Espresso machine": 8990.0, "Coffee grinder": 2450.0,
                "Filter kettle": 1290.0, "Bean subscription": 690.0,
                "Travel mug": 349.0}
    RW = {"North": 1.30, "South": 0.80, "Centre": 1.55, "West": 0.95}
    CW = {"Retail": 1.00, "Online": 1.25, "Wholesale": 2.10}
    MW = [0.72, 0.78, 0.90, 0.95, 1.00, 1.05, 0.98, 0.92, 1.08, 1.15, 1.45, 1.60]

    rows, start = [], date(2025, 1, 6)
    for week in range(52):
        day = start + timedelta(weeks=week)
        for region in REGIONS:
            for _ in range(rng.randint(1, 2)):
                product = rng.choice(list(PRODUCTS))
                channel = rng.choice(CHANNELS)
                base = 9 * RW[region] * CW[channel] * MW[day.month - 1]
                units = max(1, round(rng.gauss(base, base * 0.28)))
                price = PRODUCTS[product] * rng.choice([1.0, 1.0, 1.0, 0.9, 0.85])
                rows.append({"date": day.isoformat(), "region": region,
                             "channel": channel, "product": product,
                             "units": str(units),
                             "unit_price": f"$ {price:,.2f}"})

    # la suciedad deliberada: una región tecleada de cuatro formas,
    # celdas en blanco, y renglones capturados dos veces
    for i in rng.sample(range(len(rows)), 24):
        rows[i]["region"] = rng.choice(["north", "NORTH", " North", "North "])
    for i in rng.sample(range(len(rows)), 11):
        rows[i]["units"] = ""
    for i in rng.sample(range(len(rows)), 7):
        rows.append(dict(rows[i]))
    rng.shuffle(rows)

    with open("sales.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["date", "region", "channel",
                                          "product", "units", "unit_price"])
        w.writeheader()
        w.writerows(rows)

    with open("regions.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["region", "manager", "country", "monthly_target"])
        w.writerows([["North", "Ana Robles", "Mexico", 480000],
                     ["South", "Luis Ferrer", "Mexico", 300000],
                     ["Centre", "Paula Ines", "Mexico", 560000],
                     ["West", "Marco Duarte", "Mexico", 360000],
                     ["East", "Sofia Lara", "Mexico", 220000]])

    AREAS = {
        "Sales": (["Account executive", "Sales analyst", "Sales manager"], 24000, 62000),
        "Marketing": (["Content specialist", "Campaign analyst", "Brand manager"], 22000, 58000),
        "Finance": (["Accounts clerk", "Financial analyst", "Controller"], 26000, 74000),
        "People": (["Recruiter", "People analyst", "People manager"], 21000, 55000),
        "Operations": (["Warehouse lead", "Logistics analyst", "Operations manager"], 20000, 60000),
    }
    CITIES = ["Mexico City", "Guadalajara", "Monterrey", "Queretaro"]
    emp = []
    for n in range(1, 121):
        area = rng.choice(list(AREAS))
        titles, low, high = AREAS[area]
        idx = rng.choices([0, 1, 2], weights=[5, 3, 1])[0]
        tenure = rng.randint(2, 132)
        salary = round(low + (high - low) * (idx / 2) * rng.uniform(0.82, 1.10)
                       + tenure * 45, -2)
        emp.append({"employee_id": f"E{n:04d}", "area": area,
                    "job_title": titles[idx], "city": rng.choice(CITIES),
                    "tenure_months": tenure, "monthly_salary": int(salary)})
    with open("employees.csv", "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=list(emp[0]))
        w.writeheader()
        w.writerows(emp)


try:
    _descargar()
    print("Datos leídos del repositorio.")
except Exception:
    _reconstruir_datos()
    print("El repositorio no respondió. Datos reconstruidos en esta sesión.")

print("Listos:", ", ".join(ARCHIVOS))

### La limpieza, otra vez y sin explicación

Agrupar sobre datos sucios es el primer error de esta sesión, así que el archivo se limpia
antes de tocarlo. Si algo de esta celda no se entiende, está explicado paso por paso en la
sesión 15.2.

In [ ]:
# La limpieza de la sesión 15.2, en una celda. No es material nuevo: es para que
# este cuaderno se pueda abrir solo, sin depender de que corriste el anterior.
ventas = pd.read_csv("sales.csv").drop_duplicates()
ventas["region"] = ventas["region"].str.strip().str.title()
ventas["unit_price"] = (ventas["unit_price"]
                        .str.replace("$", "", regex=False)
                        .str.replace(",", "", regex=False)
                        .str.strip()
                        .astype(float))
ventas["date"] = pd.to_datetime(ventas["date"])
ventas = ventas.dropna(subset=["units"])
ventas["units"] = ventas["units"].astype(int)
ventas["amount"] = ventas["units"] * ventas["unit_price"]

ventas.to_csv("sales_clean.csv", index=False)
print(f"{len(ventas)} renglones limpios, total {ventas['amount'].sum():,.2f}")

---
# Bloque 1 · Agrupar

`groupby` hace exactamente lo que hace arrastrar un campo a una tabla dinámica: parte los
renglones en montones que comparten un valor, aplica un resumen a cada montón, y vuelve a
juntar los resultados como tabla.

Partir, resumir, juntar. Eso es todo.

## Ochenta líneas, u ocho

Primero la versión de la semana 14, con un diccionario y un ciclo. Corre de verdad, sobre el
archivo limpio que acabas de escribir.

In [ ]:
import csv
from collections import defaultdict

por_region_manual = defaultdict(float)

with open("sales_clean.csv", encoding="utf-8") as f:
    for registro in csv.DictReader(f):
        por_region_manual[registro["region"]] += float(registro["amount"])

for region in sorted(por_region_manual, key=por_region_manual.get, reverse=True):
    print(f"{region:8} {por_region_manual[region]:>12,.2f}")

Ahora lo mismo con pandas.

In [ ]:
por_region = ventas.groupby("region")["amount"].sum()

print(por_region.sort_values(ascending=False).round(2))

Los mismos cuatro totales, hasta el centavo. La diferencia es que uno se lee de un vistazo y
el otro hay que revisarlo renglón por renglón para creerle.

Vale la pena comprobarlo en lugar de tomármelo de palabra.

In [ ]:
iguales = all(
    round(por_region_manual[region], 2) == round(por_region[region], 2)
    for region in por_region.index
)
print("¿Coinciden los cuatro totales?", iguales)

El resultado de `groupby` es una `Series` cuyo índice es aquello por lo que agrupaste, así
que todo lo de la sesión 15.1 sigue sirviendo aquí.

In [ ]:
print("Mejor región:", por_region.idxmax())
print("Su parte del año:", f"{por_region.max() / ventas['amount'].sum():.1%}")
print()
print("Las cuatro, en miles:")
print((por_region.sort_values(ascending=False) / 1000).round(0))

## Varios resúmenes a la vez

`agg` recibe una lista de funciones y devuelve una columna por cada una. Contesta cuánto,
cuántas veces y de qué tamaño en una sola pasada.

In [ ]:
resumen = ventas.groupby("region")["amount"].agg(["sum", "count", "mean"]).round(2)

print(resumen.sort_values("sum", ascending=False))

Resúmenes distintos para columnas distintas, con los nombres que tú elijas. El patrón es
`nombre_nuevo=("columna de origen", "función")`, y es como se arma una tabla de reporte en
una sola instrucción.

In [ ]:
detalle = ventas.groupby("region").agg(
    ingreso=("amount", "sum"),
    unidades=("units", "sum"),
    ventas_hechas=("amount", "count"),
    promedio=("amount", "mean"),
).round(2)

print(detalle.sort_values("ingreso", ascending=False))

### Aquí está la historia

Fíjate bien en esa tabla antes de seguir. **North vende más que nadie, y Centre tiene la
venta promedio más alta.** North hizo 92 ventas de 47 mil en promedio; Centre hizo 70 de 56
mil.

Son dos negocios distintos con el mismo ingreso aparente, y esa diferencia no se ve en un
total. `sum` contesta magnitud y `count` contesta frecuencia, que son las dos preguntas de
la semana 9, y hace falta hacer las dos para entender qué pasó.

In [ ]:
print("Ventas hechas por región:")
print(detalle["ventas_hechas"].sort_values(ascending=False))
print()
print("Tamaño promedio de la venta:")
print(detalle["promedio"].sort_values(ascending=False).round(0))

## Dos campos de agrupación

Pásale una lista y los montones se vuelven cada combinación de los dos campos.

In [ ]:
por_region_canal = ventas.groupby(["region", "channel"])["amount"].sum().round(2)

print(por_region_canal)

Doce renglones, uno por combinación. Se lee como una lista larga, y ese es justamente el
problema que resuelve el bloque siguiente.

---
# Bloque 2 · La rejilla

`pivot_table` acomoda esos mismos números como cuadrícula, que es la forma en que los ves en
pantalla cuando abres una tabla dinámica.

Cuatro argumentos, y cada uno corresponde a algo que arrastrarías con el ratón:

| Argumento | Qué es | En la tabla dinámica |
|---|---|---|
| `index` | Lo que baja por el lado | El campo que arrastras a las filas |
| `columns` | Lo que cruza arriba | El campo que arrastras a las columnas |
| `values` | Lo que llena las celdas | El campo de valores |
| `aggfunc` | Cómo se resumen | Configuración del campo de valor |

In [ ]:
rejilla = ventas.pivot_table(
    index="region",      # lo que baja por el lado
    columns="channel",   # lo que cruza arriba
    values="amount",     # lo que llena las celdas
    aggfunc="sum",       # cómo se resumen
)

print((rejilla / 1000).round(0))

Los mismos doce números del bloque anterior, ahora legibles de un vistazo. En miles, para que
quepan.

## La trampa del `aggfunc`

**Predice antes de correr.** ¿Qué devuelve `pivot_table` si no dices `aggfunc`?

- **A.** La suma por región y canal.
- **B.** El promedio, que es lo que hace por omisión.
- **C.** El conteo de renglones.
- **D.** Un error, porque `aggfunc` es obligatorio.

In [ ]:
# FALLA A PROPÓSITO. No lanza error: da otro número, que es peor.
sin_aggfunc = ventas.pivot_table(
    index="region",
    columns="channel",
    values="amount",
)

print("Sin aggfunc, celda Centre/Online:", round(sin_aggfunc.loc["Centre", "Online"], 2))
print("Con sum,      celda Centre/Online:", round(rejilla.loc["Centre", "Online"], 2))
print()
print("¿Cuántas veces más grande es la suma?",
      round(rejilla.loc["Centre", "Online"] / sin_aggfunc.loc["Centre", "Online"], 1))

La respuesta es **B**. Por omisión `pivot_table` promedia, no suma.

Y ahí está el peligro: no lanza error, devuelve una rejilla con la misma forma y los mismos
encabezados, con números treinta veces más chicos. Si esperabas totales y no dijiste
`aggfunc`, tu reporte sale mal y se ve perfectamente bien.

El número por el que difieren no es casualidad: es cuántas ventas cayeron en esa celda.

## Los totales

In [ ]:
con_totales = ventas.pivot_table(
    index="region", columns="channel", values="amount",
    aggfunc="sum", margins=True, margins_name="Total",
)

print((con_totales / 1000).round(0))

`margins=True` agrega los totales de fila y de columna, igual que el total general de una
tabla dinámica. El número de hasta abajo a la derecha tiene que coincidir con el total de la
tabla, y comprobarlo es la forma más rápida de saber si se perdió algo por el camino.

In [ ]:
esquina = con_totales.loc["Total", "Total"]
tabla = ventas["amount"].sum()

print(f"Esquina de la rejilla: {esquina:,.2f}")
print(f"Total de la tabla:     {tabla:,.2f}")
print("¿Coinciden?", round(esquina, 2) == round(tabla, 2))

## Agrupar por tiempo

Una columna de fecha se puede agrupar por cualquier parte de sí misma. `.dt` entra en la
fecha igual que `.str` entra en el texto.

In [ ]:
ventas["month"] = ventas["date"].dt.month
mensual = ventas.groupby("month")["amount"].sum().round(2)

print((mensual / 1000).round(0))
print()
print("Mejor mes:", mensual.idxmax(), "| peor mes:", mensual.idxmin())

Diciembre manda con más del doble de casi cualquier otro mes, y julio es el más flojo. Ese
patrón solo aparece cuando agrupas: en la tabla renglón por renglón no se ve.

Ahora, cuidado con la lectura fácil. El archivo trae dentro una curva estacional que sube en
noviembre y diciembre, y aun así noviembre salió abajo. La razón es que unas pocas ventas de
máquina de espresso, que es el producto caro, pesan más que la estacionalidad de todo el
resto. Un total esconde su composición, y es exactamente por eso que `agg` con `count` al
lado de `sum` vale la pena.

In [ ]:
print("Ingreso y número de ventas por mes:")
print(ventas.groupby("month").agg(
    ingreso=("amount", "sum"),
    ventas=("amount", "count"),
    ticket=("amount", "mean"),
).round(0).sort_values("ingreso", ascending=False).head())

Diciembre tuvo un ticket promedio muy por encima del resto, no muchas más ventas. El mes no
fue mejor porque se vendiera más seguido, sino porque se vendió más caro.

Trimestre, año y día de la semana funcionan igual.

In [ ]:
por_trimestre = ventas.groupby(ventas["date"].dt.quarter)["amount"].sum().round(2)
print("Por trimestre, en miles:")
print((por_trimestre / 1000).round(0))

print()
print("Por día de la semana, en miles:")
print((ventas.groupby(ventas["date"].dt.day_name())["amount"].sum() / 1000).round(0))

El día de la semana sale con un solo valor porque el archivo se generó con una venta por
semana, todos los lunes. Es un buen recordatorio de que una agrupación no inventa variedad
donde no la hay, y de que conviene mirar el resultado antes de sacar conclusiones.

## Lo más vendido de cada grupo

Una pregunta que sale en todo reporte: qué producto vende más en cada región. Se agrupa por
los dos, se totaliza, y se toma el mayor de cada región.

In [ ]:
producto_region = ventas.groupby(["region", "product"])["amount"].sum()
mejor_por_region = producto_region.loc[producto_region.groupby("region").idxmax()]

print(mejor_por_region.round(2))

La línea del medio se lee de adentro hacia afuera: `groupby("region").idxmax()` devuelve, por
cada región, la etiqueta completa del renglón más grande, y `loc` va por esos renglones. Es
el mismo `idxmax` de la sesión 15.1, aplicado a un índice de dos niveles.

---
# Bloque 3 · Unir dos tablas

`merge` es el `BUSCARV`, con dos diferencias que importan. Trae todas las columnas de golpe
en lugar de una por fórmula, y te dice qué no encontró en vez de dejar `#N/A` regados por la
hoja.

| Modo | Qué conserva | Cuándo |
|---|---|---|
| `left` | Todos los renglones de la izquierda | El seguro, y el que imita a `BUSCARV` |
| `inner` | Solo los que coinciden | Cuando lo que no cruza no interesa |
| `right` | Todos los de la derecha | Raro, es un `left` al revés |
| `outer` | Todos los de ambos lados | Para auditar qué no coincidió |

In [ ]:
regiones = pd.read_csv("regions.csv")

print(regiones)
print()
print("Ventas:", ventas.shape, "| Regiones:", regiones.shape)

`on` nombra la columna que las dos tablas comparten. Cada renglón de `regiones` que coincida
se pega al renglón de ventas, trayendo todas sus columnas con él.

In [ ]:
unida = ventas.merge(regiones, on="region", how="left")

print("Después de la unión:", unida.shape)
print(unida[["date", "region", "amount", "manager", "monthly_target"]].head(3))

`how="left"` conserva todos los renglones de ventas, encuentre o no pareja el catálogo. Ese
es el comportamiento del `BUSCARV` y es el valor seguro: nunca pierdes una venta en silencio
porque su región faltaba en el catálogo.

Fíjate en la forma: 306 renglones antes, 306 después. Si ese número hubiera cambiado, algo
pasó que hay que entender antes de seguir.

## La auditoría, que es lo que ninguna hoja te deja hacer

In [ ]:
auditoria = ventas.merge(regiones, on="region", how="outer", indicator=True)

print(auditoria["_merge"].value_counts())

`how="outer"` conserva todo de ambos lados, y la columna `_merge` dice de dónde vino cada
renglón. Las dos direcciones se revisan por separado y significan cosas distintas.

**`right_only` en uno** significa que el catálogo tiene una región sin ninguna venta. Casi
siempre está bien: una plaza nueva, o una que cerró.

**`left_only` en cero** significa que ninguna venta quedó huérfana. Eso sí importa: una venta
con una región que el catálogo no conoce es un problema de datos que hay que reportar, no
tapar.

In [ ]:
huerfanas = auditoria[auditoria["_merge"] == "right_only"]["region"].unique()
print("Regiones del catálogo sin ninguna venta:", list(huerfanas))

sin_catalogo = auditoria[auditoria["_merge"] == "left_only"]["region"].unique()
print("Ventas con una región que el catálogo no conoce:", list(sin_catalogo) or "ninguna")

Con una fórmula tendrías que contar los `#N/A` a mano, y solo en una dirección. Aquí el
conteo viene incluido y cubre las dos.

## Usar lo que trajo la unión

Ahora que cada renglón conoce su meta, la comparación es una columna normal.

In [ ]:
mensual_region = (
    unida.assign(month=unida["date"].dt.month)
    .groupby(["region", "manager", "monthly_target", "month"])["amount"]
    .sum()
    .reset_index()
)

mensual_region["cumplio"] = mensual_region["amount"] >= mensual_region["monthly_target"]
mensual_region["avance"] = (mensual_region["amount"] / mensual_region["monthly_target"]).round(3)

print(mensual_region.head())

`reset_index` convierte el índice de la agrupación de vuelta en columnas normales. Sin él,
`region`, `manager`, `monthly_target` y `month` seguirían siendo índice y no se podrían usar
en una comparación.

Y con eso ya se puede armar el tablero.

In [ ]:
tablero = (
    mensual_region.groupby(["region", "manager"])
    .agg(meses=("cumplio", "count"),
         meses_en_meta=("cumplio", "sum"),
         avance_promedio=("avance", "mean"))
    .round(3)
    .sort_values("avance_promedio", ascending=False)
)

print(tablero)

Ninguna región llegó a su meta más de tres meses de doce, y la mejor promedia 76 % de avance.
Eso no lo dice ninguna de las dos tablas por separado: `ventas` no conoce las metas y
`regiones` no conoce las ventas. Sale de haberlas unido.

## Cuando la llave se llama distinto en cada tabla

Se nombran los dos lados. Aquí las dos se llaman `region`, así que el ejemplo renombra una de
paso para que se vea la forma.

In [ ]:
codigos = regiones.rename(columns={"region": "region_code"})
ejemplo = ventas.merge(codigos, left_on="region", right_on="region_code", how="left")

print("Unida con llaves de distinto nombre:", ejemplo.shape)
print(ejemplo[["region", "region_code", "manager"]].head(3))

Nota que quedaron las dos columnas de llave, `region` y `region_code`, con el mismo
contenido. Es lo normal, y si estorban se quitan con `drop`.

## Exportar

El análisis termina donde empezó, como un archivo que alguien más puede abrir.

In [ ]:
tablero.to_csv("tablero.csv")

print("Escrito tablero.csv")
print(open("tablero.csv", encoding="utf-8").read())

Para Excel, que es donde esto suele tener que acabar, se usa un `ExcelWriter` cuando son
varias hojas. Necesita el paquete `openpyxl`, que Colab ya trae instalado.

In [ ]:
# Si openpyxl faltara, pandas lanza ImportError nombrándolo. Colab ya lo trae.
try:
    with pd.ExcelWriter("reporte.xlsx") as writer:
        tablero.to_excel(writer, sheet_name="Tablero")
        mensual_region.to_excel(writer, sheet_name="Mensual", index=False)
        regiones.to_excel(writer, sheet_name="Regiones", index=False)
    print("Escrito reporte.xlsx con tres hojas")
except ImportError as e:
    print("No se escribió el .xlsx:", e)

Los archivos quedan en la sesión de Colab. Para bajarlos a tu máquina, el panel de archivos
de la izquierda tiene la opción de descarga en el menú de cada uno.

---
## Cuatro errores al agrupar y unir

**Agrupar sin haber limpiado.** Ocho regiones donde hay cuatro. Los totales se parten y cada
mitad se ve perfectamente razonable. Por eso la limpieza va antes en este cuaderno.

**Suponer que `pivot_table` suma.** Por omisión promedia. Ya viste el número que sale, y ya
viste que no avisa.

**Unir con `inner` sin darte cuenta.** Los renglones que no cruzan desaparecen en silencio, y
tu total baja sin que nada lo explique.

**Confiar en la unión sin auditarla.** `indicator=True` cuesta una palabra y te dice
exactamente cuántos renglones quedaron sueltos, en las dos direcciones.

---
# Ejercicios

Las soluciones están hasta abajo.

## Agrupar

### Ejercicio 1 · Tres agrupaciones sencillas

Sobre `ventas`, calcula e imprime:

1. El ingreso total por canal, ordenado de mayor a menor.
2. Cuántas unidades se vendieron de cada producto.
3. El precio unitario promedio por producto, redondeado a dos decimales.

### Ejercicio 2 · El reporte de una instrucción

Arma con `agg` una tabla por producto que traiga, con estos nombres exactos: `ingreso`,
`unidades`, `ventas`, `ticket_promedio` y `precio_promedio`. Ordénala por ingreso.

Después contesta en un comentario cuál producto conviene empujar si lo que quieres es subir
el ingreso, y cuál si lo que quieres es subir el número de ventas.

### Ejercicio 3 · La rejilla que cruza tiempo

Arma una rejilla con el mes en las filas, la región en las columnas y el ingreso en las
celdas, con totales. Divídela entre mil y redondéala para que se pueda leer.

Comprueba que la esquina coincide con el total de la tabla.

## Unir

### Ejercicio 4 · La unión auditada, al revés

Haz la unión de `regiones` contra `ventas`, o sea con `regiones` del lado izquierdo, en modo
`left`. Compara cuántos renglones salen contra la unión que hicimos en clase y explica en un
comentario por qué el número es distinto.

### Ejercicio 5 · La región inventada

Agrega a mano un renglón a una copia de `ventas` con la región `"East Coast"`, que no está en
el catálogo. Corre la auditoría y comprueba que ahora `left_only` ya no es cero.

Después di, en un comentario, qué harías con ese renglón si apareciera en tu proyecto.

### Ejercicio 6 · El mes en que cada región cumplió

Con `mensual_region`, encuentra para cada región el mes de mayor avance y el de menor.
Imprime región, mes y avance de los dos.

Pista: `idxmax` dentro de un `groupby`, como el ejercicio de lo más vendido.

## Con tus datos

### Ejercicio 7 · Contesta tu pregunta de negocio

Con tu archivo limpio, contesta la pregunta que planteaste en la semana 1 usando una
agrupación. Produce además una rejilla que cruce dos categorías, y únela con una tabla de
catálogo si tu caso lo pide.

Si hay unión, tiene que venir auditada con `indicator` y comentada.

La prueba: suma la rejilla completa y compárala con el total de la tabla. Si no coinciden,
algo se perdió.

---
## Tres ideas para llevarse

**`groupby` es la tabla dinámica.** Partir, resumir y volver a juntar. Las ochenta líneas de
la semana 14, en ocho, y con los mismos totales al centavo.

**`pivot_table` promedia por omisión.** Si querías totales tienes que decirlo, y ese olvido
produce un número treinta veces más chico que se ve perfectamente razonable.

**Una unión se audita.** `indicator` cuesta una palabra y contesta, en las dos direcciones,
qué renglones no encontraron pareja.

La siguiente sesión son gráficas. Cómo se ve un número para que alguien lo entienda sin que
se lo expliques.

---
# Soluciones

### Ejercicio 1

```python
print("Ingreso por canal:")
print((ventas.groupby("channel")["amount"].sum().sort_values(ascending=False) / 1000).round(0))

print("\nUnidades por producto:")
print(ventas.groupby("product")["units"].sum().sort_values(ascending=False))

print("\nPrecio unitario promedio por producto:")
print(ventas.groupby("product")["unit_price"].mean().round(2).sort_values(ascending=False))
```

Wholesale se lleva más de la mitad del ingreso, y no porque venda más veces sino porque cada
venta es más grande. Otra vez la misma lección: magnitud y frecuencia son dos preguntas.

### Ejercicio 2

```python
por_producto = ventas.groupby("product").agg(
    ingreso=("amount", "sum"),
    unidades=("units", "sum"),
    ventas=("amount", "count"),
    ticket_promedio=("amount", "mean"),
    precio_promedio=("unit_price", "mean"),
).round(2)

print(por_producto.sort_values("ingreso", ascending=False))

# Para subir el ingreso conviene empujar Espresso machine: es el ticket más alto
# con diferencia, así que cada venta extra pesa mucho. Para subir el número de
# ventas conviene Travel mug o Bean subscription, que son los baratos y por eso
# los que más se mueven. Son dos estrategias distintas y la tabla las separa.
```

Esta tabla es el reporte completo en una instrucción. Que los nombres de columna los elijas
tú es lo que la vuelve entregable en lugar de un paso intermedio.

### Ejercicio 3

```python
por_mes = ventas.pivot_table(
    index=ventas["date"].dt.month,
    columns="region",
    values="amount",
    aggfunc="sum",
    margins=True,
    margins_name="Total",
)

print((por_mes / 1000).round(0))

esquina = por_mes.loc["Total", "Total"]
print("\n¿La esquina coincide?", round(esquina, 2) == round(ventas["amount"].sum(), 2))
```

Se le puede pasar a `index` una Series calculada al vuelo, no solo el nombre de una columna.
Es lo que evita crear una columna `month` que después estorba.

### Ejercicio 4

```python
al_reves = regiones.merge(ventas, on="region", how="left")

print("Ventas merge regiones:", len(ventas.merge(regiones, on="region", how="left")))
print("Regiones merge ventas:", len(al_reves))
print(al_reves[al_reves["amount"].isna()][["region", "manager"]])

# Sale un renglón más, 307 contra 306. Con regiones a la izquierda, how="left"
# conserva las cinco regiones del catálogo, incluida East, que no tiene ninguna
# venta. Ese renglón aparece con todas las columnas de ventas en NaN. No es un
# error: es el catálogo diciendo que East existe y no vendió nada.
```

Cuál tabla va a la izquierda es una decisión, no un detalle. La de la izquierda es la que
tiene garantizado sobrevivir completa.

### Ejercicio 5

```python
inventada = ventas.copy()
renglon = ventas.iloc[0].copy()
renglon["region"] = "East Coast"
inventada = pd.concat([inventada, renglon.to_frame().T], ignore_index=True)

revision = inventada.merge(regiones, on="region", how="outer", indicator=True)
print(revision["_merge"].value_counts())
print("\nVentas sin región en el catálogo:")
print(revision[revision["_merge"] == "left_only"][["region", "amount"]])

# Ese renglón no se borra y no se inventa una región para él. Se reporta a quien
# capturó los datos, porque "East Coast" puede ser East mal escrito, una plaza
# nueva que nadie dio de alta, o una venta de otra empresa. Las tres se arreglan
# distinto y ninguna se arregla adivinando.
```

Este ejercicio es el que justifica auditar siempre. Un renglón en 307 no cambia el total lo
suficiente para que alguien lo note, y aun así es un dato mal.

### Ejercicio 6

```python
mejor = mensual_region.loc[mensual_region.groupby("region")["avance"].idxmax()]
peor = mensual_region.loc[mensual_region.groupby("region")["avance"].idxmin()]

print("Mejor mes de cada región:")
print(mejor[["region", "manager", "month", "avance"]].to_string(index=False))
print("\nPeor mes de cada región:")
print(peor[["region", "manager", "month", "avance"]].to_string(index=False))
```

`idxmax` sobre un `groupby` devuelve una etiqueta por grupo, y `loc` las convierte en
renglones completos. Es el mismo patrón de "lo más vendido de cada grupo", y ya que lo
reconoces vas a usarlo en casi todo reporte.

### Ejercicio 7

No hay solución publicada porque el archivo es distinto para cada quien. Se califica sobre
tres cosas: que la agrupación conteste la pregunta que planteaste y no otra, que la
auditoría venga comentada si hubo unión, y que la suma de la rejilla coincida con el total de
la tabla.